# Pronóstico del precio de bolsa de energía en Colombia

Paso 3 de 3. Los dos pasos anteriores bajan los datos (`1_descargar.py`) y arman
el panel diario (`2_construir_panel.py`). Aquí se entrenan y evalúan los dos
modelos del proyecto.

| | Modelo 1 | Modelo 2 |
|---|---|---|
| Horizonte | 1 día | 7 días |
| Pregunta | ¿cuánto costará mañana? | ¿qué tan probable es que el precio pase de cierto valor en una semana? |
| Salida | un número en COP/kWh | mediana, intervalo del 80% y probabilidades |

Son dos modelos porque las dos preguntas tienen respuestas de distinta
naturaleza. A un día el precio se puede pronosticar como número. A siete días
ese número ya no le gana a repetir el precio de hoy, pero el modelo sí sabe algo
sobre el riesgo de que el precio se dispare, y eso se aprovecha entregando
probabilidades. A quince días o más no se encontró capacidad de pronóstico; eso
se resume al final.

### Qué se predice

$$y_t^{(h)} = \log P_{t+h} - \log P_t$$

el cambio del logaritmo del precio en $h$ días. El precio se reconstruye como
$\hat P_{t+h} = P_t \, e^{\hat y}$.

Se modela el cambio y no el nivel porque, con el nivel, un modelo que solo copia
el precio de ayer ya parece muy bueno. Con el cambio, copiar el precio de hoy es
el punto de partida (el modelo "ingenuo") y cualquier mejora sobre eso es
información de verdad.

### Las ventanas de tiempo

Cada decisión se toma en una ventana distinta y ninguna mira la siguiente.

| Ventana | Fechas | Para qué |
|---|---|---|
| Selección | 2015-2019 | ordenar las variables por importancia |
| Validación | 2020-2022 | elegir cuántas variables y calibrar los intervalos |
| Prueba A | 2023-2025 | primera medición |
| Prueba B | 2026 | medición final |

2026 es la prueba más exigente: cuando se tomaron las decisiones de modelado
esos datos no existían, y además fue un año de El Niño fuerte (el precio pasó de
105 a 1.031 COP/kWh).

---
## 1. Preparación

En Colab hacen falta dos archivos además de este notebook: `features.py` y
`panel_diario.csv`. La celda los busca en la carpeta, luego en GitHub, y si no los
encuentra abre el botón para subirlos.

In [ ]:
import importlib.util, subprocess, sys

for paquete in ["xgboost", "statsmodels"]:
    if importlib.util.find_spec(paquete) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

from pathlib import Path
import urllib.request

REPO = "https://raw.githubusercontent.com/andrw4-4/pronostico-precio-bolsa-colombia/main/"

for archivo in ["features.py", "panel_diario.csv"]:
    if Path(archivo).exists():
        continue
    try:
        urllib.request.urlretrieve(REPO + archivo, archivo)
        print(f"{archivo}: bajado de GitHub")
    except Exception:
        from google.colab import files
        print(f"Sube {archivo}")
        files.upload()
print("Archivos listos")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import xgboost as xgb

import features as fe

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

FIN_SELECCION  = pd.Timestamp("2019-12-31")
INI_VALIDACION = pd.Timestamp("2020-01-01")
FIN_VALIDACION = pd.Timestamp("2022-12-31")
INI_PRUEBA     = pd.Timestamp("2023-01-01")
INI_2026       = pd.Timestamp("2026-01-01")

SEMILLA = 42
SEMILLAS = [42, 7, 123, 2024, 99]   # se promedian cinco modelos para que el resultado no dependa de una semilla

panel = fe.leer_panel("panel_diario.csv")
print(f"{panel.shape[0]} días x {panel.shape[1]} variables, "
      f"del {panel.index.min().date()} al {panel.index.max().date()}")

---
## 2. La serie que se quiere pronosticar

In [ ]:
precio = panel[fe.OBJETIVO]
log_precio = np.log(precio)
dlog = log_precio.diff()

fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
precio.plot(ax=ax[0], color="k", lw=0.7, title="Precio de bolsa (COP/kWh)")
dlog.plot(ax=ax[1], color="tab:red", lw=0.5, title="Cambio diario del logaritmo del precio")
for a in ax:
    for f in [INI_VALIDACION, INI_PRUEBA, INI_2026]:
        a.axvline(f, color="gray", ls="--", lw=0.8)
plt.tight_layout(); plt.show()

for nombre, s in [("precio", precio), ("log precio", log_precio), ("cambio del log", dlog)]:
    p = adfuller(s.dropna(), autolag="AIC")[1]
    print(f"{nombre:15s} prueba ADF: p = {p:.4f}")

La prueba ADF rechaza raíz unitaria incluso en el nivel, así que formalmente no
obligaría a diferenciar. Se diferencia de todas formas por dos razones que la
prueba no mira: el nivel tiene picos de más de 2.500 COP/kWh con una varianza que
cambia mucho entre años, y, sobre todo, la razón de la sección anterior: modelar
el cambio obliga al modelo a decir algo más que "igual que ayer".

El año de prueba merece una mirada aparte:

In [ ]:
p26 = precio[precio.index >= INI_2026].dropna()
mensual = p26.resample("MS").mean()
print(f"2026: de {p26.min():.0f} a {p26.max():.0f} COP/kWh; el mes más caro "
      f"fue {mensual.max() / mensual.min():.1f} veces el más barato\n")

tabla = pd.DataFrame({
    "precio": mensual,
    "Niño 1+2 semanal": panel["nino12_anom"].resample("MS").mean(),
    "ONI": panel["oni"].resample("MS").mean(),
}).loc["2025-11":].round(2)
tabla

En febrero de 2026 la temperatura frente a Ecuador (Niño 1+2) ya estaba casi un
grado por encima de lo normal, mientras el ONI, que es un promedio de tres meses
publicado con rezago, todavía marcaba La Niña. El precio empezó a subir en mayo.
Esa diferencia de meses es la razón por la que el panel trae la serie semanal y no
solo el ONI.

---
## 3. Limpieza: qué variables entran

Todo lo de esta sección y la siguiente está en `features.py`. Aquí se llama y se
explica.

Hay tres tipos de dato faltante y cada uno se trata distinto:

1. Eventos. Vertimientos, importaciones, exportaciones, generación solar y eólica.
   XM no publica fila cuando el evento no ocurre, y en once años ninguna de estas
   series tiene un cero. El hueco es un cero real y se llena con 0. Es la
   imputación con un valor fijo del curso de Soledad Galli (sección 5, cuaderno
   *Arbitrary Value Imputation*), solo que aquí el valor no es arbitrario: es el
   que corresponde.
2. Variables que arrancan tarde. Rellenar años antes de que existieran sería
   inventar datos. Además, una sola variable que empieza tarde le recorta el
   inicio a toda la muestra, porque el modelo necesita todas las columnas el mismo
   día. Se descartan.
3. Huecos de uno o dos días (festivos, fallas de publicación). Se llenan con el
   último valor conocido, que solo mira al pasado.

In [ ]:
# comprobación del supuesto de los eventos: si no hay ceros, el hueco es "no ocurrió"
for c in ["vertimientos", "importaciones", "exportaciones"]:
    s = panel[c]
    print(f"{c:14s} faltante {s.isna().mean():5.1%}   ceros exactos: {(s == 0).sum()}")

base, descartes = fe.preparar_panel(panel)
print()
for motivo, columnas in descartes.items():
    print(f"Fuera por {motivo}: {columnas}")
print(f"\nEntran {base.shape[1]} variables (incluye 1/volumen útil, que se agrega aquí)")

Las dos variables de despacho se dejan fuera a propósito. El precio de bolsa es
el costo marginal del despacho, así que estas variables son casi el mismo precio:

In [ ]:
c = panel[["costo_marginal_prog", fe.OBJETIVO]].dropna()
print(f"correlación entre el costo marginal programado y el precio de bolsa: "
      f"{c.iloc[:, 0].corr(c.iloc[:, 1]):.3f}")

Si entraran, el modelo tendría un error bajísimo y no aprendería nada sobre la
hidrología ni la oferta, que es justamente lo que interesa.

---
## 4. Características

Para pronosticar el día $t$ solo se usa información anterior a ese día:

| Tipo de variable | Rezago mínimo | Por qué |
|---|---|---|
| Medidas durante el día (hidrología, demanda, generación) | 1 día | se conocen al cierre |
| Ex ante (disponibilidad declarada, precio de escasez, cargos) | 0 días | se publican antes del despacho |
| Curva de mérito | 1 día | la oferta del día $t$ es lo que fija el precio de ese día |

Con cada variable se construyen rezagos (1, 2, 3, 7, 14 y 30 días), medias
móviles de 7 y 30 días y "choques": cuánto se alejó el valor de ayer de su media.
El choque es comparar una variable contra un valor de referencia, como en el
cuaderno *Comparing features to reference variable* de la sección 17 del curso
de Galli; aquí la referencia es la media reciente de la misma variable. Las
variables de día de la semana salen de la sección 15 (*date-time features*).

El detalle que más fugas causa: toda media móvil se calcula con `shift(1)` antes
del `rolling`. Una media que incluyera el día $t$ usaría el dato que se quiere
pronosticar, y eso no rompe nada, simplemente infla los resultados.

In [ ]:
X, precio, log_precio = fe.construir_features(panel)
COLUMNAS = fe.limpiar_matriz(X.loc[:FIN_SELECCION].dropna())
print(f"{X.shape[1]} características creadas, {len(COLUMNAS)} después de quitar "
      f"constantes, redundantes y combinaciones exactas")

# ejemplo con una variable: cómo se ve lo que entra al modelo para un día cualquiera
dia = pd.Timestamp("2024-03-15")
ejemplo = ["volumen_util_pct_lag1", "volumen_util_pct_ma7", "volumen_util_pct_choque7"]
print(f"\nvolumen útil el {(dia - pd.Timedelta(days=1)).date()}: "
      f"{panel.loc[dia - pd.Timedelta(days=1), 'volumen_util_pct']:.4f}")
X.loc[[dia], ejemplo].round(4)

La limpieza de la matriz se calcula solo con la ventana de selección y tiene tres
filtros. Los dos primeros son los cuadernos 1 y 2 del repo Complete-Feature-Selection:
quitar columnas constantes y quitar las redundantes por correlación (allá con
umbrales de 0,7 y 0,9; aquí 0,95, porque se quiere conservar variantes de una
misma variable que no sean casi idénticas). El tercero quita combinaciones
lineales exactas. Existe porque
`x_choque7 = x_lag1 − x_ma7` por construcción: con las tres juntas la regresión no
puede separar sus efectos, que es el problema de multicolinealidad visto en las
clases 5 a 7 de SLDA.

---
## 5. Funciones que usan los dos modelos

Selección de variables. Se ordenan las variables combinando dos criterios
calculados solo con la ventana de selección:

- Lasso: regresión con penalización L1 (clases 7 y 8 de SLDA), con las variables
  estandarizadas antes (sección 18 del curso de Galli). Ve relaciones lineales.
- Ganancia en XGBoost: cuánto mejora el ajuste cada vez que un árbol usa la
  variable (clase 15 de SLDA, módulo de árboles del ML Zoomcamp). Ve no
  linealidades e interacciones.

Se promedia el puesto de cada variable en los dos rankings para no quedarse con
los sesgos de uno solo.

La validación cruzada es temporal (`TimeSeriesSplit`): siempre entrena con el
pasado y valida con el futuro, a diferencia de la validación cruzada aleatoria de
la clase 4. Lleva un hueco de `h` días entre entrenamiento y validación, porque el
objetivo de un día mira `h` días hacia adelante.

Comparación contra el ingenuo. La prueba de Diebold-Mariano dice si la diferencia
de error entre dos pronósticos es real o puede ser ruido. Es la que usa la
librería epftoolbox (Lago et al., 2021), la referencia de este campo. Lleva la
corrección de Newey-West con `h` rezagos porque, a siete días, dos pronósticos
seguidos comparten seis días de información y no son independientes.

In [ ]:
def datos_h(h):
    return pd.concat([X[COLUMNAS], fe.objetivo(log_precio, h)], axis=1).dropna()


def ranquear(X_sel, y_sel, h):
    cv = TimeSeriesSplit(n_splits=3, gap=h)
    Xs = StandardScaler().fit_transform(X_sel)
    lasso = LassoCV(cv=cv, max_iter=20000, random_state=SEMILLA, n_jobs=-1).fit(Xs, y_sel)
    puesto_lasso = pd.Series(np.abs(lasso.coef_), index=X_sel.columns).rank(ascending=False)

    arboles = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.03,
                               subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
                               reg_lambda=3.0, random_state=SEMILLA, n_jobs=-1,
                               tree_method="hist").fit(X_sel, y_sel)
    puesto_xgb = pd.Series(arboles.feature_importances_, index=X_sel.columns).rank(ascending=False)
    return ((puesto_lasso + puesto_xgb) / 2).sort_values()


def diebold_mariano(e1, e2, h=1):
    # d > 0 cuando el modelo 1 se equivoca más que el 2
    d = (e1 ** 2 - e2 ** 2).dropna()
    n = len(d)
    gamma = [d.var(ddof=0)] + [np.cov(d[k:], d[:n - k])[0, 1] for k in range(1, h)]
    varianza = (gamma[0] + 2 * sum(gamma[1:])) / n
    if varianza <= 0:
        return np.nan, np.nan
    dm = d.mean() / np.sqrt(varianza)
    dm *= np.sqrt(max(1e-9, (n + 1 - 2 * h + h * (h - 1) / n) / n))   # ajuste para muestras chicas
    return float(dm), float(2 * (1 - stats.t.cdf(abs(dm), df=n - 1)))


def metricas(y, pred, precio_hoy, precio_futuro):
    nivel = precio_hoy * np.exp(pred)
    return {"RMSE_log": float(np.sqrt(np.mean((y - pred) ** 2))),
            "MAPE_%": float(100 * np.mean(np.abs(nivel - precio_futuro) / precio_futuro)),
            "Direccion_%": float(100 * np.mean(np.sign(pred) == np.sign(y)))}


def precios(h, indice):
    # precio del día de origen y precio h días después, alineados por origen
    hoy = precio.reindex(indice)
    futuro = precio.reindex(indice + pd.Timedelta(days=h))
    futuro.index = indice
    return hoy, futuro


ESCALERA = [12, 30, 50, 100, 200, None]   # cuántas variables probar; None = todas

Hiperparámetros. Los de los dos modelos salieron de una búsqueda aleatoria con
validación cruzada temporal sobre 2015-2022 (script `legacy/10_horizontes.py`).
Los tres que más pesan son los que explica el módulo de XGBoost del ML Zoomcamp:
la tasa de aprendizaje (`learning_rate`, allá `eta`), la profundidad de los
árboles y el tamaño mínimo de las hojas. La búsqueda eligió árboles muy poco
profundos (2 niveles) y una tasa de aprendizaje baja, o sea modelos simples y muy
regularizados. Más adelante se explica por qué eso tiene sentido aquí.

---
## 6. Modelo 1: pronóstico a un día

Pregunta: ¿cuánto costará mañana? Rival: "mañana costará lo mismo que hoy".

In [ ]:
H1 = 1
PARAMS1 = dict(n_estimators=600, max_depth=2, learning_rate=0.01, subsample=0.9,
               colsample_bytree=1.0, min_child_weight=10, reg_lambda=10.0)


def entrenar_puntual(entrena, cols):
    return [xgb.XGBRegressor(**PARAMS1, random_state=s, n_jobs=-1, tree_method="hist")
            .fit(entrena[cols], entrena["y"]) for s in SEMILLAS]


def predecir_puntual(modelos, datos, cols):
    return pd.Series(np.mean([m.predict(datos[cols]) for m in modelos], axis=0),
                     index=datos.index)


datos1 = datos_h(H1)
orden1 = ranquear(datos1.loc[:FIN_SELECCION, COLUMNAS], datos1.loc[:FIN_SELECCION, "y"], H1)
print("Las 10 variables mejor puestas para un día:")
for i, v in enumerate(orden1.index[:10], 1):
    print(f"  {i:2d}. {v}")

### Cuántas variables usar

Se prueba con 12, 30, 50, 100, 200 y todas, y se elige la mejor en la ventana de
validación. Elegirla mirando el periodo de prueba sería como probar seis modelos
y reportar el que mejor salió: alguno acierta por suerte.

In [ ]:
entrena_val = datos1.loc[:INI_VALIDACION - pd.Timedelta(days=H1 + 1)]
val = datos1.loc[INI_VALIDACION:FIN_VALIDACION]
hoy, futuro = precios(H1, val.index)

resultado = {}
for k in ESCALERA:
    cols = list(orden1.index) if k is None else list(orden1.index[:k])
    pred = predecir_puntual(entrenar_puntual(entrena_val, cols), val, cols)
    m = metricas(val["y"], pred, hoy, futuro)
    ingenuo = metricas(val["y"], pred * 0, hoy, futuro)
    resultado[len(cols)] = 100 * (m["RMSE_log"] / ingenuo["RMSE_log"] - 1)
    print(f"  {len(cols):3d} variables: error {resultado[len(cols)]:+.2f}% frente al ingenuo")

K1 = min(resultado, key=resultado.get)
cols1 = list(orden1.index[:K1])
print(f"\nLa validación elige {K1} variables")

### Medición

In [ ]:
filas = []
for etapa, ini, fin in [("2023-2025", INI_PRUEBA, INI_2026 - pd.Timedelta(days=1)),
                        ("2026", INI_2026, datos1.index.max())]:
    entrena = datos1.loc[:ini - pd.Timedelta(days=H1 + 1)]
    prueba = datos1.loc[ini:fin]
    hoy, futuro = precios(H1, prueba.index)

    modelos1 = entrenar_puntual(entrena, cols1)
    pred = predecir_puntual(modelos1, prueba, cols1)
    m = metricas(prueba["y"], pred, hoy, futuro)
    ingenuo = metricas(prueba["y"], pred * 0, hoy, futuro)
    dm, p = diebold_mariano(prueba["y"] - pred, prueba["y"], h=H1)

    filas.append({"periodo": etapa, "días": len(prueba), "variables": K1,
                  "MAPE modelo %": m["MAPE_%"], "MAPE ingenuo %": ingenuo["MAPE_%"],
                  "dirección %": m["Direccion_%"],
                  "RMSE vs ingenuo %": 100 * (m["RMSE_log"] / ingenuo["RMSE_log"] - 1),
                  "p Diebold-Mariano": p})
    if etapa == "2026":
        pred1_2026, prueba1_2026 = pred, prueba

tabla1 = pd.DataFrame(filas).set_index("periodo")
tabla1.round(4)

In [ ]:
hoy, futuro = precios(H1, prueba1_2026.index)
dia_objetivo = prueba1_2026.index + pd.Timedelta(days=H1)
estimado = hoy * np.exp(pred1_2026)

fig, ax = plt.subplots(2, 1, figsize=(13, 8))
ax[0].plot(dia_objetivo, futuro.values, color="k", lw=1, label="real")
ax[0].plot(dia_objetivo, estimado.values, color="tab:red", lw=1, label="modelo")
ax[0].plot(dia_objetivo, hoy.values, color="tab:blue", lw=0.8, ls="--", label="ingenuo")
ax[0].set_title("Modelo 1 en 2026"); ax[0].set_ylabel("COP/kWh"); ax[0].legend(fontsize=8)

ax[1].plot(dia_objetivo, (futuro - estimado).abs().rolling(21).mean().values,
           color="tab:red", label="error del modelo")
ax[1].plot(dia_objetivo, (futuro - hoy).abs().rolling(21).mean().values,
           color="tab:blue", ls="--", label="error del ingenuo")
ax[1].set_title("Error absoluto, media móvil de 21 días"); ax[1].set_ylabel("COP/kWh")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Qué variables pesan

La importancia por permutación se mide sobre 2026: cuánto empeora el error si se
desordena una variable. Es más confiable que la ganancia interna de XGBoost, que
se calcula sobre el entrenamiento y premia variables que el modelo usó mucho
aunque no generalicen.

In [ ]:
perm = permutation_importance(modelos1[0], prueba1_2026[cols1], prueba1_2026["y"],
                              n_repeats=10, random_state=SEMILLA,
                              scoring="neg_root_mean_squared_error")
importancia = pd.Series(perm.importances_mean, index=cols1).sort_values()
importancia.tail(20).plot.barh(figsize=(9, 6), color="tab:blue",
                               title="Importancia por permutación en 2026 (20 más altas)")
plt.tight_layout(); plt.show()

---
## 7. Modelo 2: pronóstico probabilístico a siete días

### Por qué cambia la pregunta

Si a este modelo se le pide un solo número para dentro de siete días, empata con
la regla ingenua (MAPE de 28,7% contra 28,7% en 2026). Medido así no sirve.

Pero el empate esconde algo: el modelo sí tiene información, solo que sobre la
dispersión y no sobre el centro. No sabe hacia dónde va la mediana, pero sí sabe
cuándo hay más riesgo de que el precio se dispare. Por eso se le pregunta otra
cosa: no "¿cuánto costará el 15 de abril?" sino "¿qué tan probable es que el
15 de abril pase de 500 COP/kWh?". Para programar un mantenimiento o comprar una
cobertura, esa es la pregunta útil.

### Cómo se construye

Un modelo por cuantil (10%, 25%, 50%, 75% y 90%), entrenado con la pérdida
pinball. Esa pérdida castiga distinto quedarse corto que pasarse: el modelo del
percentil 90 paga caro quedarse por debajo del valor real, así que aprende el
borde de arriba de la distribución.

El rival es la climatología: "el precio en siete días será el de hoy multiplicado
por un cambio sacado al azar de los cambios de siete días que se han visto". Es
la versión probabilística de repetir el precio de hoy.

In [ ]:
H7 = 7
CUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90]
NIVEL = 0.80
PARAMS7 = dict(n_estimators=400, max_depth=2, learning_rate=0.01, subsample=0.7,
               colsample_bytree=1.0, min_child_weight=10, reg_lambda=3.0)


def pinball(y, pred, q):
    d = y - pred
    return float(np.mean(np.maximum(q * d, (q - 1) * d)))


def entrenar_cuantiles(entrena, cols):
    return {q: xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q,
                                random_state=SEMILLA, n_jobs=-1, tree_method="hist",
                                **PARAMS7).fit(entrena[cols], entrena["y"])
            for q in CUANTILES}


def predecir_cuantiles(modelos, datos, cols):
    d = pd.DataFrame({q: m.predict(datos[cols]) for q, m in modelos.items()}, index=datos.index)
    d.values.sort(axis=1)   # el percentil 25 no puede quedar por encima del 50
    return d


def climatologia(entrena, prueba):
    cambios = entrena["y"].values
    return pd.DataFrame({q: np.full(len(prueba), np.quantile(cambios, q)) for q in CUANTILES},
                        index=prueba.index)


datos7 = datos_h(H7)
orden7 = ranquear(datos7.loc[:FIN_SELECCION, COLUMNAS], datos7.loc[:FIN_SELECCION, "y"], H7)

entrena_val = datos7.loc[:INI_VALIDACION - pd.Timedelta(days=H7 + 1)]
val = datos7.loc[INI_VALIDACION:FIN_VALIDACION]
perdida = {}
for k in ESCALERA:
    cols = list(orden7.index) if k is None else list(orden7.index[:k])
    q = predecir_cuantiles(entrenar_cuantiles(entrena_val, cols), val, cols)
    perdida[len(cols)] = np.mean([pinball(val["y"], q[c], c) for c in CUANTILES])
    print(f"  {len(cols):3d} variables: pérdida pinball {perdida[len(cols)]:.5f}")

K7 = min(perdida, key=perdida.get)
cols7 = list(orden7.index[:K7])
print(f"\nLa validación elige {K7} variables")

### Calibración: que el 80% signifique 80%

Los cuantiles que salen de XGBoost están mal calibrados: el intervalo que dice
80% cubre alrededor del 70% de los casos. Un intervalo así da una confianza que
no tiene.

Se corrige con predicción conforme sobre los cuantiles (*conformalized quantile
regression*, Romano, Patterson y Candès, 2019). La idea es sencilla: se mira
cuánto se equivocó el intervalo en los casos recientes ya resueltos y se ensancha
lo necesario. Para cada caso se calcula

$$E = \max\big(q_{10}(x) - y,\; y - q_{90}(x)\big)$$

que es positivo cuando el valor real quedó fuera del intervalo, y dice por cuánto.
El percentil 80 de esos errores es lo que se suma y se resta.

La versión original usa un conjunto fijo para calcular esos errores. Aquí se usa
una ventana móvil con los últimos casos, porque 2023-2026 tuvo movimientos más
grandes que 2015-2022 y una corrección fija calculada con años tranquilos se
habría quedado corta. El tamaño de la ventana se elige en validación.

Para el día $t$ solo se usan casos cuyo resultado ya se conoce: orígenes hasta
$t-7$, porque el resultado de un origen se sabe siete días después.

In [ ]:
def calibrar(q_hist, y_hist, dias, ventana):
    errores = np.maximum(q_hist[CUANTILES[0]] - y_hist, y_hist - q_hist[CUANTILES[-1]])
    ajuste = pd.Series(index=dias, dtype=float)
    for t in dias:
        ultimo = t - pd.Timedelta(days=H7)        # último origen con resultado conocido
        e = errores.loc[(errores.index > ultimo - pd.Timedelta(days=ventana))
                        & (errores.index <= ultimo)].dropna()
        if len(e) < 20:
            ajuste[t] = 0.0
            continue
        n = len(e)
        ajuste[t] = float(np.quantile(e, min(1.0, np.ceil((n + 1) * NIVEL) / n)))
    return ajuste


def prob_superar(q, ajuste, cambio_umbral):
    # P(y > umbral) interpolando entre los cuantiles ya corregidos
    niveles = np.array(CUANTILES)
    M = q[CUANTILES].values.copy()
    M[:, 0] -= ajuste.values
    M[:, -1] += ajuste.values
    M.sort(axis=1)
    u = np.asarray(cambio_umbral, dtype=float)
    return pd.Series([1 - np.interp(u[i], M[i], niveles, left=0.0, right=1.0)
                      for i in range(len(u))], index=q.index)


modelos_val = entrenar_cuantiles(entrena_val, cols7)
historia_val = datos7.loc[:FIN_VALIDACION]
q_hist_val = predecir_cuantiles(modelos_val, historia_val, cols7)
q_val = predecir_cuantiles(modelos_val, val, cols7)

cobertura = {}
for ventana in [90, 180, 365]:
    a = calibrar(q_hist_val, historia_val["y"], val.index, ventana)
    cobertura[ventana] = float(((val["y"] >= q_val[CUANTILES[0]] - a)
                                & (val["y"] <= q_val[CUANTILES[-1]] + a)).mean())
    print(f"  ventana de {ventana} días: cobertura {cobertura[ventana]:.1%}")

VENTANA = min(cobertura, key=lambda w: abs(cobertura[w] - NIVEL))
print(f"\nVentana elegida: {VENTANA} días")

### Medición

Tres medidas, cada una responde algo distinto:

- Pérdida pinball: la forma correcta de puntuar cuantiles (el RMSE no sirve aquí).
- Cobertura: ¿el intervalo del 80% contiene el 80% de los casos?
- Brier: para probabilidades del tipo "pasa de 500", mide si están bien
  calibradas, no solo si aciertan.

In [ ]:
UMBRALES = [300, 500, 800]
filas = []
for etapa, ini, fin in [("2023-2025", INI_PRUEBA, INI_2026 - pd.Timedelta(days=1)),
                        ("2026", INI_2026, datos7.index.max())]:
    entrena = datos7.loc[:ini - pd.Timedelta(days=H7 + 1)]
    prueba = datos7.loc[ini:fin]
    modelos7 = entrenar_cuantiles(entrena, cols7)

    historia = datos7.loc[:fin]
    ajuste = calibrar(predecir_cuantiles(modelos7, historia, cols7), historia["y"],
                      prueba.index, VENTANA)
    q = predecir_cuantiles(modelos7, prueba, cols7)
    clima = climatologia(entrena, prueba)
    hoy, futuro = precios(H7, prueba.index)
    abajo, arriba = q[CUANTILES[0]] - ajuste, q[CUANTILES[-1]] + ajuste

    fila = {"periodo": etapa, "días": len(prueba), "variables": K7,
            "pinball vs climatología %": 100 * (
                np.mean([pinball(prueba["y"], q[c], c) for c in CUANTILES])
                / np.mean([pinball(prueba["y"], clima[c], c) for c in CUANTILES]) - 1),
            "cobertura sin calibrar": float(((prueba["y"] >= q[CUANTILES[0]])
                                             & (prueba["y"] <= q[CUANTILES[-1]])).mean()),
            "cobertura calibrada": float(((prueba["y"] >= abajo) & (prueba["y"] <= arriba)).mean()),
            "ancho medio COP/kWh": float(np.mean(hoy * (np.exp(arriba) - np.exp(abajo))))}

    for u in UMBRALES:
        cambio = np.log(u) - np.log(hoy)
        real = (futuro > u).astype(float)
        brier_m = np.mean((prob_superar(q, ajuste, cambio) - real) ** 2)
        brier_c = np.mean((prob_superar(clima, ajuste * 0, cambio) - real) ** 2)
        fila[f"Brier >{u} vs climatología %"] = 100 * (brier_m / brier_c - 1)
    filas.append(fila)
    if etapa == "2026":
        modelos7_2026, q7_2026, ajuste7_2026, prueba7_2026 = modelos7, q, ajuste, prueba

tabla7 = pd.DataFrame(filas).set_index("periodo")
tabla7.round(4).T

In [ ]:
hoy, futuro = precios(H7, prueba7_2026.index)
dia_objetivo = prueba7_2026.index + pd.Timedelta(days=H7)
abajo = hoy.values * np.exp(q7_2026[CUANTILES[0]].values - ajuste7_2026.values)
arriba = hoy.values * np.exp(q7_2026[CUANTILES[-1]].values + ajuste7_2026.values)
mediana = hoy.values * np.exp(q7_2026[0.50].values)
p500 = prob_superar(q7_2026, ajuste7_2026, np.log(500) - np.log(hoy))

fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
ax[0].fill_between(dia_objetivo, abajo, arriba, alpha=0.3, color="tab:red", label="intervalo del 80%")
ax[0].plot(dia_objetivo, mediana, color="tab:red", lw=1, label="mediana")
ax[0].plot(dia_objetivo, futuro.values, color="k", lw=1.2, label="real")
ax[0].set_title("Modelo 2 en 2026"); ax[0].set_ylabel("COP/kWh"); ax[0].legend(fontsize=8)
ax[1].plot(dia_objetivo, p500.values, color="tab:purple", label="probabilidad de pasar de 500")
ax[1].plot(dia_objetivo, (futuro.values > 500).astype(float), color="k", lw=0.8, alpha=0.4,
           label="pasó de 500")
ax[1].set_ylabel("probabilidad"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## 8. Pronóstico desde el último día con datos

Se usan los mismos modelos que se midieron en 2026 (entrenados hasta finales de
2025), con la calibración actualizada con los casos resueltos de este año. Así
lo que se pronostica corresponde exactamente a lo que se evaluó arriba. En uso
real convendría reentrenarlos cada cierto tiempo.

In [ ]:
T = precio.dropna().index.max()            # último día con precio publicado
x_T = X.loc[[T]]
p_T = precio[T]

manana = p_T * np.exp(predecir_puntual(modelos1, x_T, cols1).iloc[0])

q_T = predecir_cuantiles(modelos7_2026, x_T, cols7)
a_T = calibrar(predecir_cuantiles(modelos7_2026, datos7, cols7), datos7["y"], [T], VENTANA)
bajo_T = p_T * np.exp(q_T[CUANTILES[0]].iloc[0] - a_T.iloc[0])
alto_T = p_T * np.exp(q_T[CUANTILES[-1]].iloc[0] + a_T.iloc[0])
med_T = p_T * np.exp(q_T[0.50].iloc[0])

print(f"Último precio conocido: {T.date()}, {p_T:.0f} COP/kWh\n")
print(f"Modelo 1, precio esperado para el {(T + pd.Timedelta(days=1)).date()}: {manana:.0f} COP/kWh\n")
print(f"Modelo 2, para el {(T + pd.Timedelta(days=7)).date()}:")
print(f"  mediana {med_T:.0f} COP/kWh, intervalo del 80% entre {bajo_T:.0f} y {alto_T:.0f}")
for u in UMBRALES:
    pr = prob_superar(q_T, a_T, [np.log(u) - np.log(p_T)]).iloc[0]
    print(f"  probabilidad de pasar de {u}: {pr:.0%}")

---
## 9. Resultados

In [ ]:
print("Modelo 1, un día")
print(tabla1.round(3).to_string())
print("\nModelo 2, siete días")
print(tabla7.round(3).to_string())

---
## 10. Lo que no funcionó

Antes de llegar a estos dos modelos se probaron varias formas de pronosticar más
allá de una semana, y ninguna funcionó. Los scripts de cada prueba están en
`legacy/` y el detalle en `DOCUMENTACION.md`.

| Intento | Resultado en 2026 |
|---|---|
| Ampliar el panel de 58 a 104 variables | sin efecto (RMSE 0,1641 frente a 0,1642) |
| El Niño semanal | corrigió el peor fallo: la dirección acertada pasó de 34% a 48% a 60 días y de 27% a 38% a 90 días, pero sin ganarle al ingenuo |
| Lluvia sobre las cuencas | sin efecto |
| Usar pocas variables | peor: el mejor número está entre 30 y 100, no en 5 |
| Historia desde 2000 (26 años, unos diez episodios de El Niño o La Niña) | no ayudó; a 30 días empeoró |
| Pronóstico puntual a 15, 30, 60 y 90 días | peor que repetir el precio de hoy en todos los casos |

Tres conclusiones de esas pruebas:

1. El horizonte útil es corto. A 30 días el modelo se equivocaba 52% contra 41% de
   repetir el precio de hoy, y a 90 días acertaba la dirección el 38% de las veces.
2. Más información no dio mejores modelos. Mezcla de generación, clima semanal y
   lluvia no movieron el resultado. La señal es débil y está repartida en muchas
   variables. Tampoco ayuda más capacidad: un modelo que memoriza el
   entrenamiento quedó 52% peor que el ingenuo en 2026, por eso no se usaron redes
   neuronales y por eso la búsqueda eligió árboles de dos niveles.
3. Resultados que se veían bien se cayeron al validarlos con cuidado. Pasó tres
   veces: los horizontes de 60 y 90 días se veían bien en 2023-2025 y fallaron en
   2026; un p-valor de 0,0496 a siete días dejó de ser significativo al promediar
   varias semillas; y elegir el número de variables mirando la prueba daba una
   mejora que desaparecía al elegirlo en validación.